In [ ]:
# --- Imports ---
import numpy as np
import xarray as xr
import frameit.utils.common as common
import matplotlib.pyplot as plt


def pressure_wind_tracker(
    mslp: xr.DataArray,
    zonal_10m: xr.DataArray,
    merid_10m: xr.DataArray,
    box_gridpoints: int,
    ):
    """
    Tracker simple du centre cyclonique :
    1) first guess = minimum de pression (MSLP) sur l'espace
    2) raffinement = minimum de vent total 10 m dans une boîte
        de taille ±box_gridpoints/2 autour du first guess.

    La fonction est vectorisée sur toutes les dimensions autres que
    les deux dernières (typiquement time, éventuellement member, etc.).
    Elle ne contient pas de boucle Python explicite.
    """

    # On suppose que les deux dernières dims sont (y, x)
    nix, njy = mslp.dims[-1], mslp.dims[-2]
    ny = mslp.sizes[njy]
    nx = mslp.sizes[nix]

    # Vérifications minimales
    if mslp.dims != zonal_10m.dims or mslp.dims != merid_10m.dims:
        raise ValueError("mslp, zonal_10m et merid_10m doivent avoir les mêmes dimensions")

    # 1) First guess : minimum global de MSLP, argmin sur les deux dims spatiales
    #    -> retourne un dict de DataArray d'indices pour chaque dim réduite
    idx_mslp = mslp.argmin(dim=[njy, nix])
    cy0 = idx_mslp[njy]  # indices en y (dims = toutes les dims sauf njy, nix)
    cx0 = idx_mslp[nix]  # indices en x (idem)

    # 2) Vent total à 10 m
    wind_10m = np.hypot(zonal_10m, merid_10m)

    # 3) Construction d'un masque "boîte" autour de (cy0, cx0) pour chaque pas de temps
    half = box_gridpoints // 2

    # Grilles d'indices 1D pour y et x
    j = xr.DataArray(
        np.arange(ny),
        dims=(njy,),
        coords={njy: mslp[njy]},
    )
    i = xr.DataArray(
        np.arange(nx),
        dims=(nix,),
        coords={nix: mslp[nix]},
    )

    # Broadcast des indices 1D et des centres (cy0, cx0) sur tous les points du champ
    # Résultat : arrays 3D (ou plus) avec mêmes dims que wind_10m
    j3, _ = xr.broadcast(j, wind_10m)
    i3, _ = xr.broadcast(i, wind_10m)
    cy0_3, _ = xr.broadcast(cy0, wind_10m)
    cx0_3, _ = xr.broadcast(cx0, wind_10m)

    # Masque de la boîte de recherche en indices (boîte carrée en indices de grille)
    mask_box = (abs(j3 - cy0_3) <= half) & (abs(i3 - cx0_3) <= half)

    # 4) On force les points hors de la boîte à une valeur de vent très grande,
    #    puis on cherche le minimum global de ce champ modifié
    max_wind = wind_10m.max(skipna=True)
    wind_boxed = wind_10m.where(mask_box, other=max_wind + 1.0)

    idx_wind = wind_boxed.argmin(dim=[njy, nix])
    cy = idx_wind[njy].astype(int)
    cx = idx_wind[nix].astype(int)

    # cy et cx sont des DataArray d'indices (dims = dims restantes, par ex. time)
    return cy, cx


In [ ]:
import logging
import os

from pathlib import Path
from frameit.core.settings_class import SimulationConfig
from frameit.core.runner import frameitRunner

#parallel computation
from multiprocessing import Pool,cpu_count

os.environ.pop("HDF5_DEBUG", None)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s",force=True)
logging.getLogger().setLevel(logging.DEBUG)

# conf = SimulationConfig.from_yaml_with_model_preset(Path("../configs_ktests/arome_oper_batsirai.yml"))
# conf = SimulationConfig.from_yaml_with_model_preset(Path("../configs_ktests/arome_oper_belna.yml"))

# conf = SimulationConfig.from_yaml_with_model_preset(Path("../configs_ktests/mnh_chido.yml"))
conf = SimulationConfig.from_yaml_with_model_preset(Path("../configs_ktests/mnh_ianos.yml"))

conf.printID()

if conf.DEBUG:
    logging.getLogger().setLevel(logging.DEBUG)
runner = frameitRunner(conf)
runner.run()

### MNH

In [ ]:
ds = runner.ds_tracker['wind_pressure']['surface']
MSLP = ds.MSLP
VM10 = ds.VM10
UM10 = ds.UM10
ds = ds.assign(Vtot=np.hypot(ds.VM10,ds.UM10))
Vtot = ds.Vtot
nj = ds.nj
ni = ds.ni
cy,cx = pressure_wind_tracker(MSLP,UM10,VM10,100)

NI = ni[cx.values]
NJ = nj[cy.values]
print(NI.values)
print(NJ.values)

In [ ]:
fig, ax = plt.subplots(ncols=2,figsize=(10, 4))
tt=3
MSLP.isel(time=tt).plot.contourf(ax=ax[0])#,xlim=(1.5e6,2.4e6),ylim=(0.4e6,1e6))
ax[0].scatter(NI[tt],NJ[tt],color='red')


UM10.isel(time=tt).plot.contourf(ax=ax[1],vmax=20)#xlim=(1.5e6,2.4e6),ylim=(0.4e6,1e6),vmax=20)
ax[1].scatter(NI[tt],NJ[tt],color='black')

### CAS AROME

In [ ]:
ds = runner.ds_tracker['wind_pressure']['surface']
prmsl = ds.prmsl
v10 = ds.v10
u10 = ds.u10
ds = ds.assign(Vtot=np.hypot(ds.v10,ds.u10))
Vtot = ds.Vtot
nj = ds.latitude
ni = ds.longitude
cy,cx = pressure_wind_tracker(prmsl,u10,v10,100)

NI = ni[cx.values]
NJ = nj[cy.values]
print(NI.values)
print(NJ.values)
fig, ax = plt.subplots(ncols=2,figsize=(10, 4))
tt=8
prmsl.isel(time=tt).plot.contourf(ax=ax[0])#,xlim=(44,50),ylim=(-11,-7.5))
ax[0].scatter(NI[tt],NJ[tt],color='red')


u10.isel(time=tt).plot.contourf(ax=ax[1],vmax=20)#,xlim=(44,50),ylim=(-11,-7.5))
ax[1].scatter(NI[tt],NJ[tt],color='black')
plt.tight_layout()